# Translation Pseudocode for Each Event
Each event has an associated notebook with a worked example. This notebook serves to string the code together for each event into a pseudocode summary.

## Synopsis for New Flow: 

Each event type has a list of cases in events.yaml. The EventTypeOperator class guides the high-level flow of the code by:  

1. Storing the observation types
2. Storing the metrics for each observation   
3. Holding the forecast data for later subsetting and processing
4. Defining how observations and forecasts interact

The forecast is then lazily loaded from whatever source (zarr or virtualizarr file) is provided. Each observation is loaded aided by an Observation class object, which stores:  

1. The metrics to use with the observation
2. The default and/or derived variables needed 
3. The source of the data
4. The modification code to process the data
5. The output Dataset format.  

>The metric storage in Observation allows for different metrics to be used for certain variables and not others in an EventType. For example, if in an augmented Heatwave event analysis, a user might want MAE for 850mb temps and MBE for 2m temperature.  

For each case in the events, the forecast data is checked out to see if the timestamps exist. If so, the unmodified variables from the forecasts are subset as are the times. At this point, the forecast data also is converted to a (init_time,valid_time,latitude,longitude) format.

Derived variables are now calculated for observation data and forecast data.   

>There are some event types that need derived variables, such as Craven-Brooks Significant Severe (CBSS), track locations of tropical disturbances, or integrated vapor transport and atmospheric river masks. Some observation data will have different derived variables to compare against (e.g. CBSS vs Practically Perfect storm report percentiles), which necessitates a directive built into each EventTypeOperator to clarify what is getting compared, and how. **Not all events use derived variables, but some events have both derived and default variables.**  

The forecast is in a singular format but might need to be transformed into one or more different formats for the event, such as (time, latitude, longitude) or (time, location). For each Observation, this transformation now occurs if needed and both the Observation and Forecast is combined into a discrete EvaluationObject that can be used to compute the metrics in that object independently.  

The metrics are then computed in this flow:  

1. EvaluationObjects are looped through
2. Metrics in each EvaluationObject are processed
3. Outputs are appended to a master DataFrame for final output with associated metadata (observation type, lead time, metric name, event type, location, case title)



#

## Heat Wave

### Before:

**Data Load and Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; generate IndividualHeatWaveCase dataclasses for subsetting  

**Data Subsetting and Processing:**  
&emsp; load forecast from source  
&emsp; load GHCN  
&emsp; load ERA5  
&emsp; subset forecast dataset using metadata  
  
&emsp; for each event_type (just heatwave for now):  
&emsp;&emsp; create the HeatWave event class with IndividualHeatWaveCase objects  

**Individual Case Evaluation:**  
&emsp; for each case in event.cases:  
&emsp;&emsp; create CaseEvaluationData object for ERA5  
&emsp;&emsp; create CaseEvaluationData object for GHCN  
  
&emsp; for each CaseEvaluationData object:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset variable (surface temp)  
&emsp;&emsp; subset time range  
&emsp;&emsp; if point obs:  
&emsp;&emsp;&emsp; filter by case id  
&emsp;&emsp;&emsp; convert longitude to 0-360  
&emsp;&emsp;&emsp; align_point_obs_from_gridded() to match gridpoints  
&emsp;&emsp;&emsp; convert from pandas dataframe to xarray dataset  

**Metric Computation:**  
&emsp; for each data variable (only one for heatwave):  
&emsp;&emsp; for each metric in metric_list:  
&emsp;&emsp;&emsp; instantiate metric()  
&emsp;&emsp;&emsp; for each observation_type in ['gridded', 'point']:  
&emsp;&emsp;&emsp;&emsp; compute metric[data_var]  
&emsp;&emsp;&emsp;&emsp; format result into dataframe

### After:

**Data Prep:**  
&emsp; get metadata from events.yaml for the cases; spatial, time, case info  
&emsp; load variable name mappings to stitch all files together  
&emsp; load EventTypeOperator object using events.yaml with heat_wave cases

**Data Load and Subsetting:**  
&emsp; load forecast from source   

&emsp; for Observation objects in EventTypeOperator:  
&emsp;&emsp; load Observation data  

&emsp; for each case in event.cases:  
&emsp;&emsp; check if forecast data exists, skip if not  
&emsp;&emsp; subset forecast variables (surface temp)  
&emsp;&emsp; subset forecast time range and convert to lead_time, valid_time

**Processing:**  
&emsp;&emsp; if variables include a DerivedVariable:  
&emsp;&emsp;&emsp; generate the variable from the Observation and forecast data with DerivedVariable  

&emsp;&emsp; for each Observation:  
&emsp;&emsp;&emsp; for each variable in Observation:  
&emsp;&emsp;&emsp;&emsp; transform subset forecast data to Observation coordinates  
&emsp;&emsp;&emsp;&emsp; combine subset forecast and Observation into EvaluationObject  


**Metric Computation:**  
&emsp; for each EvaluationObject:  
&emsp;&emsp; for each metric in EvaluationObject metric list:   
&emsp;&emsp;&emsp; compute metric  
&emsp;&emsp;&emsp; format result into dataframe  